 Lasso, ridge, elastic net regression

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Helper function to convert NOAA alphanumeric damage strings ("10K", "2.5M", "1B") to numbers
def parse_damage(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().upper()
    if val_str.endswith('K'):
        return float(val_str[:-1]) * 1_000
    elif val_str.endswith('M'):
        return float(val_str[:-1]) * 1_000_000
    elif val_str.endswith('B'):
        return float(val_str[:-1]) * 1_000_000_000
    try:
        return float(val_str)
    except ValueError:
        return 0.0

# 1. Load dataset
print("Loading capstone_2023_merged.csv...")
df = pd.read_csv("capstone_2023_merged.csv", low_memory=False)

# 2. Convert text damage fields into continuous numbers
if 'DAMAGE_PROPERTY' in df.columns:
    df['DAMAGE_PROPERTY_NUMERIC'] = df['DAMAGE_PROPERTY'].apply(parse_damage)

# 3. Target and Feature Selection
target_col = 'amountPaidOnBuildingClaim' if 'amountPaidOnBuildingClaim' in df.columns else 'amountPaidOnTotalClaim'
feature_cols = ['DAMAGE_PROPERTY_NUMERIC', 'primary_landcover_code', 'alternate_landcover_code']
feature_cols = [c for c in feature_cols if c in df.columns]

# Clean data
data = df[feature_cols + [target_col]].copy()
data[target_col] = pd.to_numeric(data[target_col], errors='coerce').fillna(0)
data = data.fillna(data.median(numeric_only=True))

X = data[feature_cols]
y_log = np.log1p(np.maximum(0, data[target_col]))

# 4. Generate Polynomial and Interaction Terms
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
poly_feature_names = poly.get_feature_names_out(feature_cols)

# 5. Train/Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X_poly, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Training Regularized Models ---")

# --- Ridge Regression ---
ridge_cv = GridSearchCV(Ridge(), param_grid={'alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]}, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
best_ridge = ridge_cv.best_estimator_
y_pred_ridge = best_ridge.predict(X_test_scaled)
ridge_r2 = r2_score(y_test, y_pred_ridge)

# --- Lasso Regression ---
lasso_cv = GridSearchCV(Lasso(max_iter=10000), param_grid={'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}, cv=5)
lasso_cv.fit(X_train_scaled, y_train)
best_lasso = lasso_cv.best_estimator_
y_pred_lasso = best_lasso.predict(X_test_scaled)
lasso_r2 = r2_score(y_test, y_pred_lasso)

# --- Elastic Net Regression ---
elastic_cv = GridSearchCV(
    ElasticNet(max_iter=10000), 
    param_grid={'alpha': [0.001, 0.01, 0.1, 1.0], 'l1_ratio': [0.2, 0.5, 0.7]}, 
    cv=5
)
elastic_cv.fit(X_train_scaled, y_train)
best_elastic = elastic_cv.best_estimator_
y_pred_elastic = best_elastic.predict(X_test_scaled)
elastic_r2 = r2_score(y_test, y_pred_elastic)

# Print Comparison Results
results = pd.DataFrame({
    'Model': ['Ridge Regression', 'Lasso Regression', 'Elastic Net'],
    'Best Parameters': [str(ridge_cv.best_params_), str(lasso_cv.best_params_), str(elastic_cv.best_params_)],
    'Test R²': [ridge_r2, lasso_r2, elastic_r2]
})

print("\n=== Week 2 Model Comparison Results ===")
print(results.to_string(index=False))

Loading capstone_2023_merged.csv...

--- Training Regularized Models ---

=== Week 2 Model Comparison Results ===
           Model                  Best Parameters  Test R²
Ridge Regression                 {'alpha': 100.0} 0.003276
Lasso Regression                 {'alpha': 0.001} 0.003339
     Elastic Net {'alpha': 0.01, 'l1_ratio': 0.2} 0.003204


In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Create the results DataFrame
data = {
    'Model': ['Lasso Regression', 'Ridge Regression', 'Elastic Net'],
    'Best Parameters': ["{'alpha': 0.001}", "{'alpha': 100.0}", "{'alpha': 0.01, 'l1_ratio': 0.2}"],
    'Test R²': [0.003339, 0.003276, 0.003204],
    'Primary Function': ['Feature selection (L1 penalty)', 'Collinearity management (L2 penalty)', 'Combined L1/L2 regularization']
}

df = pd.DataFrame(data)

# 2. Render as a table image using matplotlib
fig, ax = plt.subplots(figsize=(11, 3), dpi=300)
ax.axis('off')

# Render table
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc='center',
    loc='center'
)

# Style table
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# Highlight header
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#40466e')
        cell.set_text_props(color='white', weight='bold')
    else:
        cell.set_facecolor('#f5f5f5' if row % 2 == 0 else 'white')

# 3. Save as high-resolution image
plt.savefig('week2_model_comparison.png', bbox_inches='tight', pad_inches=0.1)
plt.close()

print("Success! Saved comparison table image as 'week2_model_comparison.png'")

Success! Saved comparison table image as 'week2_model_comparison.png'
